# <div align="center"> Statistical Learning </div>
# <div align="center"> Machine Learning and Econometrics </div>
## <div align="center"> ECO 4971/6971 </div>
#### <div align="center">Class 13 — Unsupervised Learning</div>
<div align="center"> Jonathan Holmes (he/him)</div>

# Road Map

$$Y = f(X) + \varepsilon$$

Every method we have studied so far assumes we observe the outcome $Y$. Today we drop that assumption entirely. In **unsupervised learning** there is no response variable — only a matrix of features $\mathbf{X}$. Our goal shifts from *prediction* to *discovery*: finding hidden structure in the data.

By the end of this class you will be able to:

- Explain what unsupervised learning is and why it is useful.
- Apply **K-means clustering** to partition observations into $K$ groups by minimizing within-cluster variation.
- Build and interpret a **hierarchical clustering dendrogram** without pre-specifying the number of clusters.
- Use **principal component analysis (PCA)** to reduce the dimensionality of a dataset while retaining as much variance as possible.
- Understand why and when we need to scale variables before applying these methods.

# Unsupervised Learning

Every method we have studied so far has been an example of **supervised learning**: we observed labelled pairs $\{Y_i, X_i\}_{i=1}^n$ and trained a model $\hat{f}$ to predict $Y$ from $X$. The label $Y$ *supervised* the learning — it told the algorithm what a good prediction looked like.

In **unsupervised learning** there is no response variable. We observe only the features $X_1, X_2, \dots, X_p$ for $n$ observations, and the goal is to discover interesting structure in the data: natural groupings, low-dimensional representations, or unusual patterns. Without a $Y$ to evaluate predictions against, there is no standard measure of success — which makes these methods harder to assess and more exploratory in nature.

## Why Use Unsupervised Learning?

If we have no outcome to predict, why bother? Often, the structure *in the features themselves* is exactly what we care about. Consider an e-commerce retailer trying to personalise product recommendations. The retailer has no labelled training data of the form "this customer would enjoy that product" — only a purchase and browsing history for each user. By grouping customers with similar histories together, the algorithm can recommend products that similar customers have bought, without ever observing a direct outcome. This is the logic behind collaborative filtering, the engine behind most recommendation systems.

More generally, unsupervised methods are useful whenever we want to understand the natural groupings in a dataset, compress high-dimensional data for visualisation, or detect anomalies — all without requiring labelled examples.

# Part 1: Clustering Methods

**Clustering** refers to a broad family of techniques for finding subgroups, or **clusters**, among the observations in a dataset. We seek to partition observations into distinct groups such that observations within each group are similar to one another, while observations in different groups are quite different. What "similar" means is a domain-specific choice — usually Euclidean distance in feature space, but other metrics are possible.

We will study two classical algorithms: **K-means clustering**, which requires us to specify the number of clusters $K$ in advance, and **hierarchical clustering**, which builds a tree of nested cluster merges that can be cut at any level to yield any desired number of groups.

## K-Means Clustering

**K-means clustering** partitions the dataset into $K$ distinct, non-overlapping clusters. The algorithm requires only one input from the analyst: the number of clusters $K$. Given $K$, each of the $n$ observations is assigned to exactly one of the $K$ clusters.

Before working through the math, let's see it in action on a simulated dataset where we know the true group labels — this lets us judge how well the algorithm recovers structure we built in by hand.

In [ ]:
# data
import pandas as pd
import numpy as np

# visualization
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="white")

# ML
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans

# create simulated dataset: 150 observations, 2 features, 4 true groups
X, y = make_blobs(n_samples=150, n_features=2, centers=4, cluster_std=2, random_state=17)
df = pd.DataFrame(columns=['x1', 'x2'], data=X)
df['group'] = y
df.sort_values('group', inplace=True)

df.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: unlabeled — what the algorithm actually sees
sns.scatterplot(x='x1', y='x2', color='k', s=80, data=df, ax=axes[0])
axes[0].set_title('Unlabeled data — what the algorithm sees', fontsize=12)

# Right: true labels for reference
sns.scatterplot(x='x1', y='x2', hue='group', s=80, palette='tab10', data=df, ax=axes[1])
axes[1].set_title('True group labels (hidden from the algorithm)', fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
x = df[['x1', 'x2']]

# Fit K-means with K=2 — the algorithm doesn't know the true number of groups is 4
# Try: K=3, K=4, K=5, K=6
km2 = KMeans(n_clusters=2, n_init=20, random_state=1706)
km2.fit(x)
df['Predicted Group'] = km2.labels_

fig, ax = plt.subplots(1, 1, figsize=(8, 8))
sns.scatterplot(x='x1', y='x2', s=80, hue='Predicted Group', palette='tab10', data=df, ax=ax)
ax.set_title('K-means with K=2', fontsize=12)
plt.show()

In [ ]:
k_list = [2, 3, 4]
x = df[['x1', 'x2']]
fig, axes = plt.subplots(1, len(k_list), figsize=(20, 6))

for i, k in enumerate(k_list):
    km = KMeans(n_clusters=k, n_init=20, random_state=1706)
    km.fit(x)
    pred = f'Predicted Group - k={k}'
    df[pred] = km.labels_
    sns.scatterplot(x='x1', y='x2', hue=pred, data=df, ax=axes[i], palette='tab10')
    axes[i].set_title(f'K = {k}', fontsize=12)
    if k == 4:
        km4 = km  # save the K=4 model for the decision boundary plot below

plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrix: actual group (rows) vs K=4 predicted group (columns)
print("Before label alignment:")
display(pd.crosstab(df['group'], df['Predicted Group - k=4'],
                    rownames=['Actual Group'], colnames=['Predicted Group (K=4)']))

# K-means labels are arbitrary integers — swap 0↔2 to align with the true group labels
df['Predicted Group - k=4'].replace({2: 0, 0: 2}, inplace=True)

print("After label alignment:")
display(pd.crosstab(df['group'], df['Predicted Group - k=4'],
                    rownames=['Actual Group'], colnames=['Predicted Group (K=4)']))

In [ ]:
df['Classification'] = df.apply(
    lambda r: 'Correct' if r['group'] == r['Predicted Group - k=4'] else 'Incorrect', axis=1
)

# Build a fine meshgrid covering the plot area and predict the K=4 cluster
# for every grid point — this lets us shade each decision region.
x1_min, x1_max = df['x1'].min() - 1, df['x1'].max() + 1
x2_min, x2_max = df['x2'].min() - 1, df['x2'].max() + 1
xx1, xx2 = np.meshgrid(np.linspace(x1_min, x1_max, 400),
                        np.linspace(x2_min, x2_max, 400))
Z = km4.predict(np.c_[xx1.ravel(), xx2.ravel()]).reshape(xx1.shape)

fig, ax = plt.subplots(figsize=(9, 8))

# Shade the four decision regions (boundaries are Voronoi cells of the centroids)
ax.pcolormesh(xx1, xx2, Z, cmap='Pastel1', alpha=0.5, shading='auto')

# Overlay observations: color = correct/incorrect, shape = true group
sns.scatterplot(
    x='x1', y='x2',
    hue='Classification',
    style='group',
    palette={'Correct': 'steelblue', 'Incorrect': 'tomato'},
    s=100, data=df, ax=ax
)
ax.set_title('K=4 decision regions (shape = true group, color = result)', fontsize=12)
plt.show()

## k-means explained:  notation
- Let $C_1, \dots, C_K$  denote sets containing the indices of the observations in each cluster. 
- These sets satisfy two properties:
    1. $C_1\cup C_2 \cup \dots \cup C_K = \{1, \dots, n\}$. In other words, each observation belongs to at least one of the K clusters.
    2. $C_k ∩ C_{k'} = \emptyset$ for all $k \neq k'$. In other words, the clusters are nonoverlapping: no observation belongs to more than one cluster.
    - if the $i_{th}$ observation is in the $k_{th}$ cluster, then $i \in C_k$.
- A good clustering is one for which the within-cluster variation is as small as possible
- Let the measure of within-cluster variation for cluster k be given by $W(C_k)$

## K-Means: The Minimization Problem

$$\min_{C_1, \dots, C_K} \left\{ \sum_{k=1}^K W(C_k) \right\}$$

We want to partition the observations into $K$ clusters such that the total **within-cluster variation**, summed over all $K$ clusters, is as small as possible. The most common choice of $W(C_k)$ is the **squared Euclidean distance** — conceptually similar to minimizing mean squared error, but without a response variable $Y$. We are measuring how spread out each cluster is around its own centre.

## k-means explained: Euclidean distance
$$ \large W(C_k) = \frac{1}{\left|C_k\right|} \sum_{i, i' \in C_k} \sum_{j=1}^p(x_{ij} - x_{i'j})^2$$
- where $\left|C_k\right|$ denotes the number of observations in the $k_{th}$ cluster. 
- This is the sum of all of the pairwise squared Euclidean distances between the observations in the $k_{th}$ cluster, divided by the total number of observations in the $k_{th}$ cluster.


In [ ]:
import matplotlib.image as mpimg
img = mpimg.imread('euclidian_distance.png')
plt.imshow(img)

## k-means explained: solution
Combining the last two equations yields the optimization problem that defines K-means clustering:

$$\min_{C_1, \dots, C_K} \left\{ \sum_{k=1}^K \frac{1}{\left|C_k\right|} 
\sum_{i, i' \in C_k} \sum_{j=1}^p(x_{ij} - x_{i'j})^2 \right\}$$

## K-Means: The Algorithm

The K-means algorithm is simple and fast:

1. **Random initialization.** Randomly assign each of the $n$ observations a label from $1$ to $K$. These are the initial cluster assignments.
2. **Iterate until convergence.** Repeat the following two steps until the cluster assignments stop changing:
   - (a) *Centroid step.* For each of the $K$ clusters, compute the cluster **centroid** — the vector of feature means for all observations currently in that cluster.
   - (b) *Assignment step.* Reassign each observation to the cluster whose centroid is closest in Euclidean distance.

Each step is guaranteed to decrease the objective, so the algorithm always converges. However, it may converge to a **local minimum** rather than the global minimum. In practice we run the algorithm multiple times from different random starts (`n_init` in scikit-learn) and keep the solution with the lowest total within-cluster variation.

## Why the Algorithm Works

The key insight is that within-cluster variation around the centroid is equivalent (up to a factor of 2) to the sum of pairwise distances within the cluster:

$$ \frac{1}{\left|C_k\right|} \sum_{i, i' \in C_k} \sum_{j=1}^p(x_{ij} - x_{i'j})^2 = 
2 \sum_{i \in C_k} \sum_{j=1}^p(x_{ij} - \bar{x}_{kj})^2$$

where $\bar{x}_{kj} = \frac{1}{|C_k|} \sum_{i \in C_k} x_{ij}$ is the mean of feature $j$ in cluster $C_k$. This identity shows why the algorithm makes progress at each iteration: the centroid step minimizes the right-hand side over all possible centroids, and the assignment step minimizes it over all cluster assignments. Together, each full iteration can only decrease the total within-cluster variation.

![](sphx_glr_plot_kmeans_digits_001.png)
Source: https://scikit-learn.org/stable/auto_examples/cluster/plot_kmeans_digits.html

## In-Class Exercise #1

**K-Means Clustering**

Q1: In the K = 2 solution above, the algorithm splits the data into two groups even though the true data has four clusters. Why does K-means produce a result like this when $K$ is misspecified? What would you expect to happen to total within-cluster variation as $K$ increases toward 4?

Q2: The K-means algorithm starts from a random initialisation. Why might two runs with the same $K$ return different cluster assignments? How does the `n_init` parameter address this?

Q3: Modify the code to try $K = 5$ and $K = 6$. Does the solution look sensible? What does this suggest about the challenge of choosing $K$?

In [ ]:
# Your code here


# Part 2: Hierarchical Clustering

A key limitation of K-means clustering is that we must commit to a number of clusters $K$ before seeing the data. **Hierarchical clustering** avoids this by building a complete tree of nested cluster merges, called a **dendrogram**, which can then be "cut" at any height to produce any desired number of clusters.

The most common variant is **agglomerative** (bottom-up) clustering. We start by treating every observation as its own cluster (the leaves of the tree), then repeatedly merge the two most similar clusters until only one cluster remains (the trunk). The result is a dendrogram that encodes the full merge history — like the decision tree structures we saw in Class 11, except here the tree is built bottom-up from the data.

### Linkage: measuring distance between groups

When merging clusters, we need a way to measure the distance between two *groups* of observations, not just two individual points. The choice of **linkage method** affects the shape of the dendrogram:

| Linkage | Distance between clusters $A$ and $B$ |
| --- | --- |
| **Complete** | Maximum pairwise distance: $\max_{i \in A,\, i' \in B} d(i, i')$ |
| **Single** | Minimum pairwise distance: $\min_{i \in A,\, i' \in B} d(i, i')$ |
| **Average** | Mean pairwise distance: $\frac{1}{\lvert A \rvert \cdot \lvert B \rvert}\sum_{i \in A}\sum_{i' \in B} d(i, i')$ |

Complete and average linkage tend to produce more balanced, compact dendrograms and are the most widely used in practice. We use complete linkage below.

## The Algorithm: Building the Tree One Merge at a Time

The agglomerative algorithm is straightforward. Starting with every observation as its own cluster, it repeatedly finds the two closest clusters and merges them:

1. **Start.** Treat each of the $n$ observations as its own cluster. You now have $n$ clusters, each containing a single point.

2. **Find the closest pair.** Using your chosen linkage method, compute the distance between every pair of clusters. Identify the two that are closest.

3. **Merge.** Combine those two clusters into one. You now have $n - 1$ clusters. Record the merge height — the distance at which the merge happened — on the dendrogram.

4. **Repeat.** Recompute distances between the newly merged cluster and all remaining clusters, then go back to step 2.

5. **Stop.** Continue until all observations have been merged into a single cluster. The dendrogram now shows the complete history of every merge from step 1 to step $n - 1$.

To make this concrete: suppose you have five observations — A, B, C, D, E. In the first step the algorithm might find that A and B are closest and merge them into cluster {A, B}. Now you have four clusters: {A, B}, C, D, E. Next it might find that D and E are closest and merge them into {D, E}. Then it merges C into {A, B} to get {A, B, C}. Eventually {A, B, C} and {D, E} merge into one final cluster containing everyone. Each of these merges appears as a horizontal line in the dendrogram, drawn at the height corresponding to the distance at which that merge occurred.

In [ ]:
# create simulated dataset based on two features and 4 distinct groups
X, y = make_blobs(n_samples=45, n_features=2, centers=3, cluster_std=2 , random_state=1)
df=pd.DataFrame(columns=['x1','x2'], data=X)
df['group']=y
display(df.head())

txt="Forty-five observations generated in two-dimensional space.\nIn reality there are three distinct classes, shown in separate colors.\nHowever, we will treat these class labels as unknown and will seek to cluster\nthe observations in order to discover the classes from the data."
fig, ax =  plt.subplots(1,1, figsize=(8,8))
sns.scatterplot(x='x1',y='x2',hue='group',s=80,legend=False,data=df)
fig.text(0.5, -.05, txt, ha='center', wrap=True, fontsize=12)

plt.show()

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram

mergings = linkage(df[['x1', 'x2']], method='complete')

fig, ax = plt.subplots(1, 1, figsize=(10, 10))
dendrogram(mergings, leaf_rotation=90, leaf_font_size=10, ax=ax)

# Y-axis = the distance at which two clusters were merged (complete linkage:
# the maximum pairwise distance between any two points in the merging clusters).
# Higher merges = more dissimilar clusters being joined.
ax.set_ylabel('Merge distance (complete linkage)', fontsize=12)

plt.show()

In [ ]:

    
fig, ax =  plt.subplots(1,1, figsize=(8,8))
sns.scatterplot(x='x1',y='x2',hue='group',alpha=0.5,legend=False,data=df)
for i in df.index:
    ax.annotate(i, (df.x1.loc[i], df.x2.loc[i]), ha='center')
plt.show()

## Interpreting a Dendrogram

Each **leaf** at the bottom of the dendrogram represents a single observation. As we move up the tree, leaves begin to fuse into **branches** — each fusion corresponds to a merge of two clusters. Two key rules:

- **Height of fusion = dissimilarity.** The higher up in the tree that two observations first meet, the more dissimilar they are. Observations that fuse early (near the bottom) are very similar; those that fuse late (near the top) are quite different.
- **Cutting the tree gives clusters.** Drawing a horizontal line at height $h$ produces as many clusters as there are vertical lines crossing that cut. Choosing the cut height plays the same role as choosing $K$ in K-means.

A useful heuristic for choosing the cut: look for the largest vertical gap in the dendrogram that is *not* already crossed by a horizontal line — that gap suggests where a natural number of clusters lies. Cross-referencing with the scatter plot above (where we labeled the observations by their index) lets you verify whether the clusters make geometric sense.

## Cutting the Dendrogram to Get Cluster Labels

Reading the dendrogram visually is useful, but in practice we need to actually extract cluster assignments from it. The `fcluster` function from scipy does this: it "cuts" the dendrogram at a chosen level and returns a group label for each observation.

The most intuitive option is `criterion='maxclust'`, which lets you specify the number of clusters you want — just like $K$ in K-means. The function works out where to draw the cut automatically.

Try changing `n_clusters` below and re-running the cell to see how the groupings change.

In [ ]:
from scipy.cluster.hierarchy import fcluster

# Interactive: change n_clusters to 2, 3, or 4 and re-run
n_clusters = 3

# Cut the dendrogram to extract a label (1 … n_clusters) for each observation
df['HC Group'] = fcluster(mergings, t=n_clusters, criterion='maxclust')

# Find the height of the cut: midpoint between the merge that creates n_clusters
# groups and the one just above it (which would create n_clusters - 1 groups)
heights = sorted(mergings[:, 2], reverse=True)
cut_height = (heights[n_clusters - 2] + heights[n_clusters - 1]) / 2

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Left: dendrogram with the cut line drawn in red
dendrogram(mergings, leaf_rotation=90, leaf_font_size=10, ax=axes[0])
axes[0].axhline(cut_height, color='red', linestyle='--', linewidth=1.5,
                label=f'Cut → {n_clusters} clusters')
axes[0].set_ylabel('Merge distance (complete linkage)', fontsize=12)
axes[0].set_title('Dendrogram with cut', fontsize=12)
axes[0].legend(fontsize=11)

# Right: scatter plot coloured by the resulting group labels
sns.scatterplot(x='x1', y='x2', hue='HC Group', palette='tab10',
                s=80, data=df, ax=axes[1])
axes[1].set_title(f'Predicted groups — hierarchical clustering, K={n_clusters}', fontsize=12)

plt.tight_layout()
plt.show()

## In-Class Exercise #2

**Hierarchical Clustering**

Q1: In the dendrogram above, identify two observations that are very similar (fuse near the bottom of the tree) and two that are very dissimilar (fuse near the top). Cross-reference with the labelled scatter plot — do these pairs look close or far apart in feature space?

Q2: At approximately what height would you cut the dendrogram to recover three clusters? How does this partition compare to K-means with $K = 3$?

Q3: We used *complete linkage* above. Change `method='complete'` to `method='single'` and replot the dendrogram. How does the shape change, and why? (Hint: think about what single linkage does when one observation is close to a large cluster.)

In [ ]:
# Your code here


# Part 3: Principal Component Analysis

So far in this class we have studied methods that predict an outcome $Y$, and clustering methods that group observations. **Principal component analysis (PCA)** does something different: it compresses a dataset with many variables into a smaller number of new variables — called **principal components** — that still capture most of the information.

Think of it this way. Suppose you survey 50 countries and measure GDP growth, consumption growth, investment growth, and export growth. These four variables tend to move together — when an economy is booming, all four rise. PCA would find a single "economic activity" score that summarises most of the variation across all four variables, rather than forcing you to track all four separately.

More generally, PCA is useful when:
- You have many variables and want a compact summary for visualisation or analysis.
- Your variables are correlated with each other (so they carry redundant information).
- You want to understand *which* variables drive the most variation in the data.

## What Are Principal Components?

Imagine you have 50 observations measured on 10 variables. You could look at all the pairwise scatterplots to explore the data — but there are 45 of them, and each one only shows two variables at a time.

PCA instead asks: *is there a single new variable — a weighted combination of the original ten — that captures as much of the overall variation as possible?* That new variable is the **first principal component**. A second component then captures as much of the *remaining* variation as possible, and so on.

The end result is a small number of new variables (the principal components) that you can plot and interpret, while still reflecting the main patterns in all the original data. The pair plot below for `USArrests` gives a sense of what the raw data looks like before we apply PCA.

In [ ]:
df=pd.read_csv("USArrests.csv", index_col=0)
display(df.head())
df.info()

In [ ]:
sns.pairplot(df)

## The First Principal Component

The first principal component $Z_1$ is a **weighted sum** of the original variables:

$$Z_1 = \phi_{11}X_1 + \phi_{21}X_2 + \dots + \phi_{p1}X_p$$

The coefficients $\phi_{11}, \phi_{21}, \dots, \phi_{p1}$ are called **loadings**. They tell us how much each original variable contributes to this new summary variable. A large positive loading means the variable moves strongly in the same direction as the component; a large negative loading means it moves in the opposite direction; a loading near zero means the variable barely contributes.

One constraint is imposed: the loadings must satisfy $\sum_{j=1}^p \phi_{j1}^2 = 1$ (their squared values sum to one). This keeps the scale of $Z_1$ well-defined — without it, we could make the variance of $Z_1$ arbitrarily large just by multiplying all the weights by a big number.

Among all possible weighted sums satisfying this constraint, the first principal component is the one with the **largest variance** — it is the direction along which the data spread out the most.

## Computing the First Principal Component

Before running PCA we centre each variable to have mean zero (we only care about how observations *differ* from each other, not the overall level). The **score** of observation $i$ on the first component is then:

$$z_{i1} = \phi_{11}x_{i1} + \phi_{21}x_{i2} + \dots + \phi_{p1}x_{ip}$$

Finding the best loadings is an optimisation problem — we want the weights $\phi_{11}, \dots, \phi_{p1}$ that make the variance of these scores as large as possible:

$$\max_{\phi_{11}, \dots, \phi_{p1}} \left\{\frac{1}{n} \sum_{i=1}^n \left( \sum_{j=1}^p \phi_{j1}x_{ij} \right)^2 \right\} \quad \text{s.t.} \quad \sum_{j=1}^p\phi_{j1}^2=1$$

In plain English: *find the set of weights that spreads the data out as much as possible along one direction, subject to the constraint that the weights don't just blow up to infinity.* Scikit-learn solves this automatically using a standard linear algebra technique.

## Geometric Interpretation

Geometrically, the first principal component is a line drawn through the cloud of data points that captures the most spread. Each observation's score $z_{i1}$ is just its position along that line — how far it sits from the centre in the direction of maximum variation.

The figure below shows this for a simple two-variable case. The solid line is the first principal component. Notice that the data spread out much more along this line than along any other direction.

![](figure6_4.png)
*The first principal component (solid line) is the direction of greatest spread. Each point's score is its coordinate along this axis.*

## The Second Principal Component

Once we have the first component, we build the second one using the same idea — find the weighted sum that captures as much of the *remaining* variation as possible. The only additional requirement is that the second component must be **uncorrelated with the first**.

Geometrically, this means the second component must point at right angles (perpendicular) to the first. In the two-variable case that is easy to picture: if the first component runs diagonally across a scatter plot, the second runs in the perpendicular direction.

$$z_{i2} = \phi_{12}x_{i1} + \phi_{22}x_{i2} + \dots + \phi_{p2}x_{ip}$$

We can keep building components this way — each one perpendicular to all previous ones — until we have as many components as we have variables. In practice, the first two or three components usually explain most of the variation, and that is all we need.

## Illustration: USArrests

We will apply PCA to the `USArrests` dataset, which records four variables for each of the 50 U.S. states:

- **Murder**, **Assault**, **Rape**: arrests per 100,000 residents for each crime.
- **UrbanPop**: the percentage of the state population living in urban areas.

The goal is to replace these four variables with just two principal components and plot all 50 states on a single chart, while still capturing the main patterns in the data. We will walk through each step of the process below.

In [ ]:
# Step 1: Look at the raw data
# Notice that Assault has a much larger mean and std than the other variables.
# If we ran PCA on the raw data, Assault would dominate just because of its scale,
# not because it's genuinely more informative. We need to fix this before running PCA.
display(df.head())
df.describe().T.loc[:, ['mean', 'std']]

In [ ]:
from sklearn.preprocessing import scale

# Step 2: Scale each variable to mean 0 and standard deviation 1.
# After scaling, a one-unit change means the same thing for every variable,
# so PCA treats them on equal footing.
X = pd.DataFrame(scale(df), index=df.index, columns=df.columns)
X.describe().T.loc[:, ['mean', 'std']]  # all means ≈ 0, all stds ≈ 1

### Step 3: Compute the loading vectors

The **loading table** below shows the weights that PCA assigned to each variable for each principal component. Each column is one component; each row is one variable. A large absolute value means that variable contributes heavily to that component.

For example, if Murder, Assault, and Rape all have large loadings on PC1, it suggests PC1 is capturing an overall "crime rate" dimension — states with high scores on PC1 tend to have high rates across all three crimes.

In [ ]:
from sklearn.decomposition import PCA

# Fit PCA and extract the loading vectors.
# .components_ has shape (n_components, n_features); we transpose so rows = variables.
pca_loadings = pd.DataFrame(
    PCA().fit(X).components_.T,
    index=df.columns,
    columns=[r'$\phi_1$', r'$\phi_2$', r'$\phi_3$', r'$\phi_4$']
)
pca_loadings

In [ ]:
# Verify the constraint: squared loadings for each component must sum to 1.
# This confirms PCA found proper unit-length directions.
np.square(pca_loadings).sum().reset_index().rename(
    columns={'index': 'Component', 0: 'Sum of squared loadings'}
)

In [ ]:
# Step 4: Compute the principal component scores for every state.
# fit_transform(X) applies the loadings to X and returns each state's coordinates
# in the new PC space. PC1 and PC2 are what we will plot.
pca = PCA()
df_plot = pd.DataFrame(
    pca.fit_transform(X),
    columns=['PC1', 'PC2', 'PC3', 'PC4'],
    index=X.index
)
df_plot

## How Many Components Do We Need?

After fitting PCA, we want to know how much information each component actually captures. The **proportion of variance explained** (PVE) answers this: it tells us what fraction of the total variation in the data is accounted for by each principal component.

The **scree plot** displays this visually — PVE on the y-axis, component number on the x-axis. We look for an "elbow" where the line flattens out; components beyond that point are adding little new information. The cumulative plot (right panel) shows how much total variation we have captured by the time we include each component.

In [ ]:
pve = pd.DataFrame({
    'PC': [f'PC{i+1}' for i in range(4)],
    'PVE': pca.explained_variance_ratio_,
    'Cumulative PVE': pca.explained_variance_ratio_.cumsum()
})
display(pve)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].bar(pve['PC'], pve['PVE'])
axes[0].set_ylabel('Proportion of Variance Explained', fontsize=12)
axes[0].set_title('Scree Plot', fontsize=12)

axes[1].plot(pve['PC'], pve['Cumulative PVE'], marker='o')
axes[1].set_ylabel('Cumulative PVE', fontsize=12)
axes[1].set_title('Cumulative Variance Explained', fontsize=12)
axes[1].set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 12))

ax1.set_xlim(-3.5, 3.5)
ax1.set_ylim(-3.5, 3.5)

# Note: the sign of a principal component is arbitrary — flipping all scores
# and loadings for a component together gives an equally valid solution.
# We plot them as-is from sklearn with no sign adjustments.

# Plot PC1 and PC2 scores for each state
for i in df_plot.index:
    ax1.annotate(i, (df_plot.PC1.loc[i], df_plot.PC2.loc[i]), ha='center', color='darkgreen')

# Reference lines at the origin
ax1.hlines(0, -3.5, 3.5, linestyles='dotted', colors='grey')
ax1.vlines(0, -3.5, 3.5, linestyles='dotted', colors='grey')

ax1.set_xlabel('First Principal Component', fontsize=16)
ax1.set_ylabel('Second Principal Component', fontsize=16)

# Overlay loading vectors on a second axis (scaled to [-1, 1])
ax2 = ax1.twinx().twiny()
ax2.set_ylim(-1, 1)
ax2.set_xlim(-1, 1)
ax2.tick_params(axis='y', colors='orange')
ax2.set_xlabel('Principal Component loading vectors', color='darkorange')

# Small offset so arrow tip and label don't overlap
a = 1.07
for i in pca_loadings[[r'$\phi_1$', r'$\phi_2$']].index:
    ax2.annotate(i, (pca_loadings[r'$\phi_1$'].loc[i] * a, pca_loadings[r'$\phi_2$'].loc[i] * a), color='darkorange')

# Draw all four loading vectors
ax2.arrow(0, 0, pca_loadings[r'$\phi_1$'][0], pca_loadings[r'$\phi_2$'][0], color='darkorange')
ax2.arrow(0, 0, pca_loadings[r'$\phi_1$'][1], pca_loadings[r'$\phi_2$'][1], color='darkorange')
ax2.arrow(0, 0, pca_loadings[r'$\phi_1$'][2], pca_loadings[r'$\phi_2$'][2], color='darkorange')
ax2.arrow(0, 0, pca_loadings[r'$\phi_1$'][3], pca_loadings[r'$\phi_2$'][3], color='darkorange')

txt = ("Biplot for the USArrests data. Green labels show the first two principal component scores for each state.\n"
       "Orange arrows show the loading vectors for all four variables (axes on top and right).\n"
       "The direction and length of each arrow indicate how strongly that variable loads onto each component.")
fig.text(0.5, -0.03, txt, ha='center', wrap=True, fontsize=12)

plt.show()

## Why Did We Scale the Variables?

The figure above shows what happens when we apply PCA to the *unscaled* `USArrests` data. Because `Assault` has a much larger variance than the other variables (assault arrests are far more common than murders or rapes), it completely dominates the first principal component — the other three variables contribute almost nothing. After scaling each variable to have mean zero and standard deviation one, the principal components reflect genuine structure in the data rather than arbitrary differences in measurement units or scale.

**Rule of thumb:** always scale your variables before applying PCA unless they are already measured in the same units and you have a specific reason to let high-variance features dominate.

**Try it:** Re-run the PCA cells above after commenting out the `scale()` call. Compare the loading vectors — notice how `Assault` swamps the first component without scaling.

## Interpreting Principal Components

Once you have your loading table, a natural next step is to ask: *what does each principal component actually represent?* Researchers often try to give principal components meaningful names by looking at which variables load heavily onto them.

Look back at the loading table for `USArrests`:

- **PC1**: Murder, Assault, and Rape all have large positive loadings (around 0.54), while UrbanPop's loading is smaller (around 0.28). PC1 is picking up a general **violent crime rate** — states with high PC1 scores tend to have high rates across all three crime types.
- **PC2**: UrbanPop has by far the largest loading (around 0.87), while the crime variables load much more weakly. PC2 is capturing something closer to **degree of urbanisation** — independent of overall crime levels.

We can read this directly off the biplot too: the Murder, Assault, and Rape arrows all point roughly in the same direction (along PC1), while the UrbanPop arrow points in a quite different direction (more along PC2).

The bar chart below makes the loading pattern easy to see at a glance.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, col, title in zip(axes, [r'$\phi_1$', r'$\phi_2$'], ['PC1 — "Violent Crime"', 'PC2 — "Urbanisation"']):
    bars = ax.bar(pca_loadings.index, pca_loadings[col],
                  color=['tomato' if v >= 0 else 'steelblue' for v in pca_loadings[col]])
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_ylabel('Loading', fontsize=12)
    ax.set_title(title, fontsize=12)
    ax.set_ylim(-1, 1)

plt.suptitle('Loadings for the first two principal components (USArrests)', fontsize=13)
plt.tight_layout()
plt.show()

## Benefits and Warnings

**Why interpretation is useful.** Giving principal components meaningful names makes them much easier to communicate and use. Rather than saying "a state's PC1 score is 2.3", you can say "this state has a particularly high violent crime rate relative to the national average." That is a statement a policymaker or journalist can actually act on. Interpretation also helps you check whether the results make intuitive sense — if the "crime" component had loaded heavily onto UrbanPop and barely onto Murder, you would want to go back and investigate why.

**Warning 1: Signs are arbitrary.** As we noted earlier, the sign of a principal component has no intrinsic meaning. PCA could equally well have returned a "low violent crime" PC1 with all loadings negated. The *pattern* of which variables load together is meaningful; the *direction* (positive vs. negative) is not. Be careful when comparing loadings across different software packages or re-runs — the signs may flip.

**Warning 2: Interpretation is subjective and post-hoc.** You are reading meaning into a mathematical object that was constructed purely to maximise variance — it has no built-in knowledge of economics, criminology, or anything else. Two analysts looking at the same loadings might give a component different names. This does not mean interpretation is useless, but it should be treated as an informed hypothesis rather than a fact.

**Warning 3: Correlation is not causation.** Variables loading onto the same component move together in the data, but that does not mean they are causally linked. In `USArrests`, Murder, Assault, and Rape all load onto PC1 — but this does not tell us *why* high-murder states also tend to have high assault rates. There could be a common cause (poverty, inequality, policing levels), or the pattern could be partly coincidental. PCA is a descriptive tool; causal claims require a separate analysis.

**Warning 4: Garbage in, garbage out.** If the variables you feed into PCA are poorly chosen or measure the same thing in slightly different ways, the components will reflect the structure of your variable list rather than any real underlying phenomenon. Always think carefully about what you are measuring before interpreting the results.

## In-Class Exercise #3

**Principal Component Analysis**

Q1: Look at the `pca_loadings` table. Murder, Assault, and Rape all load heavily onto the first principal component, while UrbanPop loads more onto the second. What does each principal component appear to be measuring in terms of the original variables?

Q2: In the biplot, states like Florida, Nevada, and California appear far to the right (high PC1 scores). What does a high PC1 score imply about a state's crime profile?

Q3: The scree plot shows that PC1 alone explains a large fraction of the variance. If you were building a regression model to predict something about U.S. states, at what point would you stop adding principal components? What criterion would you use?

In [ ]:
# Your code here
